# Faster Whisper 語音辨識 → 字幕檔（Colab 版）

依序執行下面三個區塊：

1. **安裝套件**：每次開新的執行階段跑一次。請先到「執行階段 → 變更執行階段類型」選 **T4 GPU**。
2. **掛載雲端硬碟**（選用）：檔案放在 Google Drive、或想把結果存回 Drive 時才需要。
3. **語音辨識**：設定參數後執行。

「來源」欄位可以填：
* 留空 → 跳出上傳視窗，從電腦選檔（可多選）
* Colab 裡的檔案路徑，例如 `/content/drive/MyDrive/錄音/0308.mp3`
* 資料夾路徑 → 辨識資料夾裡所有影音檔
* `.txt` 清單 → 一行一個路徑或網址
* YouTube 影片／播放清單網址，或其他 yt-dlp 支援的網址

繁體中文：建議勾選「轉成繁體」（用 OpenCC 轉換），**提示語留空**。提示語填指令式句子（例如「輸出為繁體中文」）容易讓辨識內容出錯或重複。


In [ ]:
#@title 1. 安裝套件 { display-mode: "form" }
!pip -q install -U faster-whisper pysubs2 yt-dlp opencc-python-reimplemented
# CTranslate2 需要 CUDA 12 的 cuBLAS / cuDNN 9，Colab 內建的版本不一定對得上
!pip -q install nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

import torch
if torch.cuda.is_available():
  print('✅ GPU:', torch.cuda.get_device_name(0))
else:
  print('⚠️ 沒有 GPU，會用 CPU 跑（很慢）。請到「執行階段 → 變更執行階段類型」選 T4 GPU 後重新執行。')


In [ ]:
#@title 2. 掛載雲端硬碟（選用） { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 3. 語音辨識 { display-mode: "form" }

#@markdown ### 來源
#@markdown 留空 = 從電腦上傳；也可填檔案路徑、資料夾、.txt 清單、YouTube 或其他網址
source = "" #@param {type:"string"}

#@markdown ### 辨識設定
language = "zh" #@param ["自動判斷", "zh", "en", "ja", "ko", "fr", "de", "es"] {allow-input: true}
model_size = "large-v3" #@param ["large-v3", "turbo", "large-v2", "medium", "small", "base"]
#@markdown 中文結果轉成台灣繁體（OpenCC s2twp）
traditional_chinese = True #@param {type:"boolean"}

#@markdown ### 輸出
output_format = "srt" #@param ["srt", "txt", "srt+txt"]
output_dir = "/content/output" #@param {type:"string"}
#@markdown 已經有結果檔的就跳過
skip_existing = False #@param {type:"boolean"}
#@markdown 完成後自動下載到電腦（多個檔案會打包成 zip）
auto_download = True #@param {type:"boolean"}
#@markdown 即時顯示辨識出來的每一句
show_segments = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 進階（通常不用改）
#@markdown 提示語：建議留空；可放專有名詞、人名，例如「寶可夢、皮卡丘」
initial_prompt = "" #@param {type:"string"}
vad_filter = True #@param {type:"boolean"}
vad_min_silence_ms = 500 #@param {type:"integer"}
beam_size = 5 #@param {type:"integer"}

import ctypes
import glob
import os
import site
import re
import subprocess
import urllib.request
import zipfile

import ctranslate2
import numpy as np
import pysubs2
from faster_whisper import WhisperModel
from google.colab import files
from tqdm.auto import tqdm
from yt_dlp import YoutubeDL

MEDIA_EXT = {'.mp3', '.wav', '.m4a', '.flac', '.aac', '.ogg', '.opus', '.wma',
             '.mp4', '.mov', '.avi', '.mkv', '.webm', '.m4v', '.3gp'}
DOWNLOAD_DIR = '/content/_downloads'
UPLOAD_DIR = '/content/uploads'


def safe_name(name):
  """把標題轉成合法檔名"""
  name = re.sub(r'[\\/:*?"<>|\r\n\t]', '_', name).strip(' .')
  return name[:150] or 'untitled'


def is_url(s):
  return re.match(r'^https?://', s) is not None


def expand_sources(src):
  """把使用者輸入展開成 [(本機路徑或網址), ...]"""
  src = src.strip()
  if not src:
    os.makedirs(UPLOAD_DIR, exist_ok=True)
    print('>> 請選擇要上傳的影音檔（可多選）')
    uploaded = files.upload()
    paths = []
    for name, data in uploaded.items():
      path = os.path.join(UPLOAD_DIR, name)
      with open(path, 'wb') as f:
        f.write(data)
      paths.append(path)
    return paths

  if is_url(src):
    return [src]

  if os.path.isdir(src):
    paths = sorted(p for p in glob.glob(os.path.join(src, '*'))
                   if os.path.splitext(p)[1].lower() in MEDIA_EXT)
    if not paths:
      print(f'⚠️ 資料夾裡沒有影音檔: {src}')
    return paths

  if src.lower().endswith('.txt'):
    if not os.path.exists(src):
      print(f'⚠️ 找不到清單檔: {src}')
      return []
    base = os.path.dirname(src)
    items = []
    with open(src, encoding='utf-8') as f:
      for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
          continue
        # 相對路徑以清單檔所在資料夾為準
        if not is_url(line) and not os.path.isabs(line) and not os.path.exists(line):
          line = os.path.join(base, line)
        items.extend(expand_sources(line))
    return items

  if os.path.exists(src):
    return [src]

  print(f'⚠️ 找不到檔案: {src}')
  return []


def download_url(url):
  """下載網址的音訊，回傳 [(音訊路徑, 標題), ...]（播放清單會有多個）"""
  os.makedirs(DOWNLOAD_DIR, exist_ok=True)

  m = re.search(r'(?:voca\.ro|vocaroo\.com)/(\w+)', url)
  if m:
    path = os.path.join(DOWNLOAD_DIR, f'vocaroo-{m.group(1)}.mp3')
    urllib.request.urlretrieve(f'https://media.vocaroo.com/mp3/{m.group(1)}', path)
    return [(path, f'vocaroo-{m.group(1)}')]

  opts = {
      'quiet': True,
      'no_warnings': True,
      'format': 'bestaudio/best',
      'outtmpl': os.path.join(DOWNLOAD_DIR, '%(id)s.%(ext)s'),
      'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'mp3'}],
      'overwrites': True,
      'ignoreerrors': True,
  }
  with YoutubeDL(opts) as ydl:
    info = ydl.extract_info(url, download=True)
  if info is None:
    return []

  entries = info.get('entries') or [info]
  results = []
  for e in entries:
    if not e:
      continue
    downloads = e.get('requested_downloads') or []
    path = downloads[0].get('filepath') if downloads else None
    if path and os.path.exists(path):
      results.append((path, e.get('title') or e.get('id')))
    else:
      print(f"⚠️ 下載失敗: {e.get('title') or e.get('webpage_url')}")
  return results


def preload_cuda_libs():
  """先載入 pip 裝的 CUDA 12 函式庫，讓 CTranslate2 找得到 libcublas.so.12 / libcudnn*.so.9"""
  patterns = ['nvidia/cuda_runtime/lib/libcudart.so.12',
              'nvidia/cublas/lib/libcublas*.so.12',
              'nvidia/cudnn/lib/libcudnn*.so.9']
  libs = [lib for d in site.getsitepackages() for p in patterns
          for lib in sorted(glob.glob(os.path.join(d, p)))]
  # 函式庫之間有相依，載入失敗的下一輪再試，直到沒有進展
  while libs:
    failed = []
    for lib in libs:
      try:
        ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
      except OSError:
        failed.append(lib)
    if len(failed) == len(libs):
      break
    libs = failed


def get_model():
  """載入模型；同一個執行階段重複執行時沿用已載入的模型"""
  global _whisper_model, _whisper_model_key
  if ctranslate2.get_cuda_device_count() > 0:
    preload_cuda_libs()
    device, compute_type = 'cuda', 'float16'
  else:
    device, compute_type = 'cpu', 'int8'
  key = (model_size, device, compute_type)
  if globals().get('_whisper_model_key') != key:
    print(f'>> 載入模型 {model_size}（{device}, {compute_type}），第一次會先下載...')
    _whisper_model = WhisperModel(model_size, device=device, compute_type=compute_type)
    _whisper_model_key = key
  return _whisper_model


def output_paths(title):
  name = safe_name(title)
  exts = ['srt', 'txt'] if output_format == 'srt+txt' else [output_format]
  return [os.path.join(output_dir, f'{name}.{ext}') for ext in exts]


def load_audio(path):
  """用 ffmpeg 解碼成 16kHz 單聲道（不經過 PyAV，避免新版 av 跟 faster-whisper 不相容）"""
  cmd = ['ffmpeg', '-nostdin', '-loglevel', 'error', '-i', path,
         '-vn', '-f', 's16le', '-ac', '1', '-ar', '16000', '-']
  proc = subprocess.run(cmd, capture_output=True)
  if proc.returncode != 0:
    raise RuntimeError('ffmpeg 解碼失敗: ' + proc.stderr.decode('utf-8', 'ignore').strip())
  return np.frombuffer(proc.stdout, np.int16).astype(np.float32) / 32768.0


def transcribe(audio_path, title, converter):
  model = get_model()
  segments, info = model.transcribe(
      load_audio(audio_path),
      language=None if language == '自動判斷' else language,
      initial_prompt=initial_prompt.strip() or None,
      beam_size=beam_size,
      vad_filter=vad_filter,
      vad_parameters=dict(min_silence_duration_ms=vad_min_silence_ms),
  )
  print(f'   語言: {info.language}（{info.language_probability:.0%}）  長度: {info.duration/60:.1f} 分鐘')

  use_opencc = converter is not None and info.language == 'zh'
  results = []
  with tqdm(total=round(info.duration), unit='秒', disable=show_segments) as pbar:
    for seg in segments:
      text = seg.text.strip()
      if use_opencc:
        text = converter.convert(text)
      if not text:
        continue
      results.append({'start': seg.start, 'end': seg.end, 'text': text})
      if show_segments:
        print(f'   [{seg.start:7.2f} → {seg.end:7.2f}] {text}')
      pbar.update(max(0, round(seg.end) - pbar.n))

  saved = []
  for path in output_paths(title):
    if path.endswith('.srt'):
      pysubs2.load_from_whisper(results).save(path, encoding='utf-8')
    else:
      with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(r['text'] for r in results) + '\n')
    saved.append(path)
  return saved


# ── 主流程 ─────────────────────────────────────────────
os.makedirs(output_dir, exist_ok=True)

converter = None
if traditional_chinese:
  import opencc
  converter = opencc.OpenCC('s2twp')

sources = expand_sources(source)
if not sources:
  print('沒有要辨識的檔案。')

finished = []
for i, src in enumerate(sources, 1):
  print(f'\n[{i}/{len(sources)}] {src}')
  try:
    if is_url(src):
      print('>> 下載中...')
      items = download_url(src)
    else:
      items = [(src, os.path.splitext(os.path.basename(src))[0])]

    for audio_path, title in items:
      outs = output_paths(title)
      if skip_existing and all(os.path.exists(p) for p in outs):
        print(f'>> 已存在，跳過: {title}')
        finished.extend(outs)
        continue
      print(f'>> 辨識中: {title}')
      finished.extend(transcribe(audio_path, title, converter))
  except Exception as e:
    print(f'❌ 失敗: {type(e).__name__}: {e}')

print('\n' + '=' * 50)
if finished:
  print(f'✅ 完成 {len(finished)} 個檔案，存放在 {output_dir}')
  for p in finished:
    print('   ', p)
  if auto_download:
    if len(finished) == 1:
      files.download(finished[0])
    else:
      zip_path = '/content/字幕結果.zip'
      with zipfile.ZipFile(zip_path, 'w') as z:
        for p in finished:
          z.write(p, os.path.basename(p))
      files.download(zip_path)
  else:
    print('請到左側「檔案」欄位下載。')
else:
  print('❌ 沒有產生任何結果檔。')
